In [1]:
import pandas as pd
import numpy as np
from numpy import str_
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

In [2]:
#Cargar el archivo que contiene los datos de los tres tipos de espectros
# (Simbioticas-1, Nebulosas-2, Gigantes Rojas-3) con DATOS COMPLETOS

# data = pd.read_csv('../../../new/datasets/unbalanced_30000.csv', header=None)
# data = pd.read_csv('../../../new/datasets/unbalanced_15000.csv', header=None)
data = pd.read_csv('../../../new/datasets/unbalanced.csv', header=None)

In [3]:
# Separamos las características y la variable objetivo
X = data.drop(data.columns[-1],  axis=1)
y = data.iloc[:, -1]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5, stratify=y)

In [5]:
# Creamos el modelo de clasificación de Random Forest
rf = RandomForestClassifier(n_estimators=500, random_state=42)

In [6]:
# Creamos el modelo de clasificación de Random Forest
# rf = RandomForestClassifier(n_estimators=500, random_state=42)
# scores = cross_val_score(rf, X, y, cv=5)
# print("Precisión: %0.2f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))

In [7]:
# Entrenamos el modelo utilizando el conjunto de entrenamiento
rf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=500, random_state=42)

In [8]:
# Hacemos predicciones en el conjunto de prueba
y_pred = rf.predict(X_test)

#Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[ 31   4   5]
 [  6  90  19]
 [  7   5 228]]


In [9]:
print("Cohen's kappa: ", cohen_kappa_score(y_test,y_pred))
print()
print(classification_report(y_test, y_pred, labels=[1,2,3], digits=4))

Cohen's kappa:  0.779490291262136

              precision    recall  f1-score   support

           1     0.7045    0.7750    0.7381        40
           2     0.9091    0.7826    0.8411       115
           3     0.9048    0.9500    0.9268       240

    accuracy                         0.8835       395
   macro avg     0.8395    0.8359    0.8353       395
weighted avg     0.8857    0.8835    0.8828       395



In [10]:
# Calculamos la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)

Accuracy: 0.8835443037974684


# Suspected Symbiotic Stars v1

In [11]:
from sklearn.metrics import confusion_matrix

df_sus_sy = pd.read_csv("../../../new/candidate_symbiotic_stars_v1/normalized/Suspected_SY.csv", header=None)

y_pred = rf.predict(df_sus_sy)
cm = confusion_matrix([1 for _ in range(len(df_sus_sy))], y_pred, labels=[1, 2, 3])
print(cm)

[[12  2  1]
 [ 0  0  0]
 [ 0  0  0]]


In [12]:
df_sus_sy_normalized = pd.read_csv("../../../new/candidate_symbiotic_stars_v1/calibrated_data/Suspected_SY.csv")
df_sus_sy_normalized['label'] = y_pred
df5 = df_sus_sy_normalized[['source_id', 'label']]
df5.head(5)

,source_id,label
0,4687286621186701568,1
1,4651824725526390016,1
2,3321366590173335424,1
3,5410876219867043072,1
4,3575939163051304192,1


In [13]:
df6 = pd.read_csv('../../../new/candidate_symbiotic_stars_v1/built_dataset/suspected_SY_dataset.csv')
df6.head(5)

,FIND_NAME,MAIN_ID,OTYPE,SP_TYPE,ID_Gaia,IDS,OTYPES,Gaia DR3
0,RAW 1691,LIN 521,C*,C,Gaia DR2 4687286621186701568,RAW 1691|LIN 521|2MASS J01183570-7242213|OGLE ...,C*|Em*|LP*|LP*|Em*|MIR|NIR|*|C*?|LP?,4687286621186701568
1,[BE74] 583,[BE74] 583,LongPeriodV*,G/Ke:,Gaia DR2 4651824725526390016,2MASS J05265014-7106350|EROS2-star lm058-2n-25...,LP*|Em*|NIR|V*|*,4651824725526390016
2,StHA 55,EM* StHA 55,Mira,NaN,Gaia DR3 3321366590173335424,IRAS 05440+0642|ASAS J054642+0643.7|ASAS J0546...,Mi*|LP*|V*|LP*|SB*|LP*|MIR|V*|Em*|NIR|*|C*?|IR...,3321366590173335424
3,ZZ CMi,V* ZZ CMi,LongPeriodV*,M6I-IIep,Gaia DR3 3155368612444708096,BD+09 1633|AN 306.1934|DO 2156|GCRV 4915|G...,LP*|NIR|V*|*|IR|LP?,3155368612444708096
4,WRAY 16−51,WRAY 16-51,LongPeriodV*,M4,Gaia DR2 5410876219860836224,IRAS 09316-4621|AKARI-IRC-V1 J0933295-463450|D...,LP*|NIR|MIR|Em*|PN|*|IR,5410876219867043072


In [14]:
# Filtro de data frames
df_filtered = df6.merge(df5, left_on=['Gaia DR3'], right_on=['source_id'], how='inner', indicator=True)
df_filtered = df_filtered[df_filtered['_merge'] == 'both']
df_filtered = df_filtered.iloc[:, [0, 1, 2, 7, 9]]
df_filtered.head(5)

,FIND_NAME,MAIN_ID,OTYPE,Gaia DR3,label
0,RAW 1691,LIN 521,C*,4687286621186701568,1
1,[BE74] 583,[BE74] 583,LongPeriodV*,4651824725526390016,1
2,StHA 55,EM* StHA 55,Mira,3321366590173335424,1
3,WRAY 16−51,WRAY 16-51,LongPeriodV*,5410876219867043072,1
4,NSV 05572,V* VX Crv,Mira,3575939163051304192,1


In [15]:
import os

out_name = 'rf_unbalanced.csv'
out_dir = '../../../new/candidate_symbiotic_stars_v1/output'
if not os.path.exists(out_dir):
    os.mkdir(out_dir)

fullname = os.path.join(out_dir, out_name)
df_filtered.to_csv(fullname, header=True, index=False)

# Suspected Symbiotic Stars v2

In [16]:
from sklearn.metrics import confusion_matrix

df_sus_sy = pd.read_csv("../../../new/candidate_symbiotic_stars_v2/normalized/Suspected_SY.csv", header=None)

y_pred = rf.predict(df_sus_sy)
cm = confusion_matrix([1 for _ in range(len(df_sus_sy))], y_pred, labels=[1, 2, 3])
print(cm)

[[13  2  2]
 [ 0  0  0]
 [ 0  0  0]]


In [17]:
df_sus_sy_normalized = pd.read_csv("../../../new/candidate_symbiotic_stars_v2/calibrated_data/Suspected_SY.csv")
df_sus_sy_normalized['label'] = y_pred
df5 = df_sus_sy_normalized[['source_id', 'label']]
df5.head(5)

,source_id,label
0,6204217186929931520,2
1,4061952680197028224,1
2,670455944074475008,1
3,4068755633500598272,3
4,2060829659152816768,2


In [18]:
df6 = pd.read_csv('../../../new/candidate_symbiotic_stars_v2/built_dataset/suspected_SY_dataset.csv')
df6.head(5)

,FIND_NAME,MAIN_ID,OTYPE,SP_TYPE,ID_Gaia,IDS,OTYPES,Gaia DR3
0,V748 Cen,V* V748 Cen,EclBin,Ae,Gaia DR3 6204217186929931520,CD-32 10517|ALS 18924|CRTS J145936.6-332503|CS...,EB*|Ro*|NIR|V*|Em*|*,6204217186929931520
1,WRAY 16-294,WRAY 16-294,LongPeriodV*,K5,Gaia DR2 4061952680197028224,2MASS J17391381-2538050|DENIS J173913.8-253805...,LP*|PN|NIR|Em*|*|C*?|ISM|LP?,4061952680197028224
2,DASCH J075731.1+201735,ASAS J075731+2017.6,LongPeriodV*,M0III,Gaia DR2 670455944074475008,2MASS J07573112+2017347|ASAS J075731+2017.6|DA...,SB*|LP*|NIR|V*|*|Opt,670455944074475008
3,ASAS J174600-2321.3,ASAS J174600-2321.3,LongPeriodV*_Candidate,F0I,Gaia DR2 4068755633500598272,2MASS J17460018-2321163|ASAS J174600-2321.3|ER...,NIR|V*|*|LP?,4068755633500598272
4,IPHASJ201550.96+373004.2,IRAS 20140+3720,PlanetaryNeb_Candidate,NaN,Gaia DR2 2060829659152816768,2MASS J20155096+3730042|AKARI-IRC-V1 J2015509+...,NIR|*|C*?|IR|LP?|PN?,2060829659152816768


In [19]:
# Filtro de data frames
df_filtered = df6.merge(df5, left_on=['Gaia DR3'], right_on=['source_id'], how='inner', indicator=True)
df_filtered = df_filtered[df_filtered['_merge'] == 'both']
df_filtered = df_filtered.iloc[:, [0, 1, 2, 7, 9]]
df_filtered.head(5)

,FIND_NAME,MAIN_ID,OTYPE,Gaia DR3,label
0,V748 Cen,V* V748 Cen,EclBin,6204217186929931520,2
1,WRAY 16-294,WRAY 16-294,LongPeriodV*,4061952680197028224,1
2,DASCH J075731.1+201735,ASAS J075731+2017.6,LongPeriodV*,670455944074475008,1
3,ASAS J174600-2321.3,ASAS J174600-2321.3,LongPeriodV*_Candidate,4068755633500598272,3
4,IPHASJ201550.96+373004.2,IRAS 20140+3720,PlanetaryNeb_Candidate,2060829659152816768,2


In [20]:
import os

out_name = 'rf_unbalanced.csv'
out_dir = '../../../new/candidate_symbiotic_stars_v2/output'
if not os.path.exists(out_dir):
    os.mkdir(out_dir)

fullname = os.path.join(out_dir, out_name)
df_filtered.to_csv(fullname, header=True, index=False)

## Other stars

In [11]:
from sklearn.metrics import confusion_matrix

df_sus_sy = pd.read_csv("../../../new/other_stars/normalized/Suspected.csv", header=None)

y_pred = rf.predict(df_sus_sy)
cm = confusion_matrix([1 for _ in range(len(df_sus_sy))], y_pred, labels=[1, 2, 3])
print(cm)

[[3 3 1]
 [0 0 0]
 [0 0 0]]


In [12]:
df_sus_sy_normalized = pd.read_csv("../../../new/other_stars/calibrated_data/Suspected.csv")
df_sus_sy_normalized['label'] = y_pred
df5 = df_sus_sy_normalized[['source_id', 'label']]
df5.head(5)

,source_id,label
0,3321366590173335424,1
1,4557410314849153920,3
2,2022052808961769088,2
3,4052553745525657600,2
4,4050670827750135040,2


In [13]:
df6 = pd.read_csv('../../../new/other_stars/symbad/suspected.csv')
df6.head(5)

,FIND_NAME,MAIN_ID,OTYPE,SP_TYPE,ID_Gaia,IDS,OTYPES,Gaia DR3
0,StHa55,EM* StHA 55,Mira,NaN,Gaia DR3 3321366590173335424,IRAS 05440+0642|ASAS J054642+0643.7|ASAS J0546...,Mi*|LP*|V*|LP*|SB*|LP*|MIR|V*|Em*|NIR|*|C*?|IR...,3321366590173335424
1,GH Gem,V* GH Gem,Symbiotic*,NaN,Gaia DR3 3160625132721733888,SV* SON 3566|AN 241.1943|ATO J106.0532+12.05...,Sy*|V*|NIR|V*|*|Opt,3160625132721733888
2,V503Her,V* V503 Her,Symbiotic*,NaN,Gaia DR3 4557410314849153920,SV* P 4385|AN 170.1936|ASAS J173640+2318.2|A...,LP*|Sy*|V*|Pu*|NIR|V*|*|Opt,4557410314849153920
3,Hen2-442,Hen 2-442,PlanetaryNeb,NaN,Gaia DR2 2022052808961769088,Hen 2-442|AKARI-IRC-V1 J1939433+262933|GSC2 N0...,PN|NIR|MIR|V*|*|IR,2022052808961769088
4,V850 Aql,V* V850 Aql,Symbiotic*,O-rich,Gaia DR2 4263728319553777408,SV* SON 4396|CSV 4646|IRAS 19210+0032|2MASS...,Sy*|Em*|Mas|AB*|LP*|NIR|PN|V*|*|IR|LP?|Mi?,4263728319553777408


In [14]:
# Filtro de data frames
df_filtered = df6.merge(df5, left_on=['Gaia DR3'], right_on=['source_id'], how='inner', indicator=True)
df_filtered = df_filtered[df_filtered['_merge'] == 'both']
df_filtered = df_filtered.iloc[:, [0, 1, 2, 7, 9]]
df_filtered.head(5)

,FIND_NAME,MAIN_ID,OTYPE,Gaia DR3,label
0,StHa55,EM* StHA 55,Mira,3321366590173335424,1
1,V503Her,V* V503 Her,Symbiotic*,4557410314849153920,3
2,Hen2-442,Hen 2-442,PlanetaryNeb,2022052808961769088,2
3,Hen2-379,PN M 1-44,PlanetaryNeb,4052553745525657600,2
4,AS288,PN H 2-43,PlanetaryNeb,4050670827750135040,2


In [15]:
import os

out_name = 'rf_unbalanced.csv'
out_dir = '../../../new/other_stars/output'
if not os.path.exists(out_dir):
    os.mkdir(out_dir)

fullname = os.path.join(out_dir, out_name)
df_filtered.to_csv(fullname, header=True, index=False)